# Treinamento e Avaliacao dos Modelos de Machine Learning
## Pipeline de Risco de Credito

**Autora:** Nayane Araujo  
**GitHub:** [Nayanearaujo](https://github.com/Nayanearaujo)  

---

### O que vamos fazer aqui?

Este e o notebook mais importante do projeto. Aqui a gente vai:

1. Carregar os dados preparados no Notebook 02
2. Treinar tres modelos de Machine Learning:
   - Regressao Logistica (baseline simples e interpretavel)
   - Random Forest (ensemble robusto)
   - XGBoost (estado da arte para dados tabulares)
3. Avaliar cada modelo com metricas de credito (AUC-ROC, F1, KS Statistic)
4. Comparar os modelos e escolher o melhor
5. Interpretar os resultados com graficos profissionais
6. Gerar insights de negocio acionaveis

### Por que essas metricas?

Em problemas de credito, **acuracia nao e suficiente**. O que importa e:

- **AUC-ROC**: o modelo consegue separar adimplentes de inadimplentes? (1.0 = perfeito, 0.5 = chute aleatorio)
- **F1-Score**: equilibrio entre precisao (nao acusar inocentes) e recall (nao deixar passar caloteiros)
- **KS Statistic**: quanto as distribuicoes de score de adimplentes e inadimplentes se separam? (padrao bancario)
- **Precision**: dos que o modelo apontou como inadimplentes, quantos realmente sao?
- **Recall**: dos inadimplentes reais, quantos o modelo conseguiu pegar?

---

## 1. Importacoes e Configuracao

In [ ]:
import warnings
import sys
import pickle
from pathlib import Path
from time import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)
from scipy.stats import ks_2samp
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print('Ambiente configurado!')

## 2. Carregamento dos Dados Preparados

In [ ]:
TRAIN_PATH = Path('../data/silver/train_prepared.csv')
TEST_PATH  = Path('../data/silver/test_prepared.csv')
MODELS_PATH = Path('../src/models')
DOCS_PATH = Path('../docs')

# Verifica se os dados preparados existem
if not TRAIN_PATH.exists():
    print('Dados preparados nao encontrados.')
    print('Execute o Notebook 02 primeiro!')
else:
    df_train = pd.read_csv(TRAIN_PATH)
    df_test  = pd.read_csv(TEST_PATH)

    y_train = df_train.pop('loan_status')
    y_test  = df_test.pop('loan_status')

    X_train = df_train.values
    X_test  = df_test.values
    feature_names = list(df_train.columns)

    print('Dados carregados com sucesso!')
    print(f'Treino : {X_train.shape[0]:,} registros x {X_train.shape[1]} features')
    print(f'Teste  : {X_test.shape[0]:,} registros x {X_test.shape[1]} features')
    print(f'Target treino: {y_train.value_counts().to_dict()}')
    print(f'Target teste : {y_test.value_counts().to_dict()}')

## 3. Definicao dos Modelos

Vamos treinar tres modelos com complexidades diferentes:

**Regressao Logistica**: o modelo mais simples. Calcula a probabilidade de inadimplencia baseado em uma combinacao linear das features. E rapido, interpretavel e serve como linha de base (baseline). Se um modelo mais complexo nao superar a Regressao Logistica, algo esta errado.

**Random Forest**: cria centenas de arvores de decisao e combina os resultados. Robusto, lida bem com nao-linearidades e valores atipicos.

**XGBoost**: gradient boosting. Constroi arvores sequencialmente, onde cada arvore corrige os erros da anterior. E o modelo preferido para dados tabulares em competicoes de ML e no mercado financeiro.

In [ ]:
# Configuracao dos modelos
modelos = {
    'Regressao_Logistica': LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',  # penaliza mais os erros na classe minoritaria
        C=0.1                      # regularizacao para evitar overfitting
    ),
    'Random_Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1                  # usa todos os nucleos da CPU
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='auc',
        use_label_encoder=False,
        n_jobs=-1
    )
}

print('Modelos configurados:')
for nome, modelo in modelos.items():
    print(f'  {nome}: {type(modelo).__name__}')

## 4. Treinamento e Avaliacao

In [ ]:
resultados = []
modelos_treinados = {}

print('Treinando modelos...')
print('=' * 70)

for nome, modelo in modelos.items():
    inicio = time()
    modelo.fit(X_train, y_train)
    tempo_treino = time() - inicio

    # Predicoes
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    # Metricas
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    ks_stat, _ = ks_2samp(y_proba[y_test == 1], y_proba[y_test == 0])

    resultado = {
        'Modelo': nome.replace('_', ' '),
        'AUC-ROC': round(auc, 4),
        'F1-Score': round(f1, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'KS Statistic': round(ks_stat, 4),
        'Tempo (s)': round(tempo_treino, 1)
    }
    resultados.append(resultado)
    modelos_treinados[nome] = (modelo, y_pred, y_proba)

    print(f'\n{nome}:')
    print(f'  AUC-ROC  : {auc:.4f}')
    print(f'  F1-Score : {f1:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall   : {recall:.4f}')
    print(f'  KS Stat  : {ks_stat:.4f}')
    print(f'  Tempo    : {tempo_treino:.1f}s')
    print(f'  Treinado com {len(X_train):,} registros')

print('\n' + '=' * 70)
print('Todos os modelos treinados!')

In [ ]:
# Tabela comparativa
df_resultados = pd.DataFrame(resultados).set_index('Modelo')

# Destaca o melhor valor em cada metrica
styled = df_resultados.style \
    .highlight_max(subset=['AUC-ROC', 'F1-Score', 'Precision', 'Recall', 'KS Statistic'],
                  color='#d4edda') \
    .highlight_min(subset=['Tempo (s)'], color='#d4edda') \
    .format(precision=4)

print('Comparacao de Modelos (verde = melhor valor por metrica):')
print(df_resultados.to_string())

# Salva na camada Gold
df_resultados.to_csv('../data/gold/model_comparison.csv')
print('\nResultados salvos em data/gold/model_comparison.csv')

## 5. Visualizacoes de Performance

Numeros sozinhos nao contam a historia completa. Vamos visualizar a performance de cada modelo.

In [ ]:
# Curvas ROC de todos os modelos
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cores_modelos = {
    'Regressao_Logistica': '#3498db',
    'Random_Forest': '#2ecc71',
    'XGBoost': '#e74c3c'
}

# Plot 1: Curvas ROC
for nome, (modelo, y_pred, y_proba) in modelos_treinados.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    label_nome = nome.replace('_', ' ')
    axes[0].plot(fpr, tpr, lw=2.5, color=cores_modelos[nome],
                 label=f'{label_nome} (AUC={auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Modelo Aleatorio (AUC=0.5000)')
axes[0].fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=11)
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=11)
axes[0].set_title('Curva ROC - Comparacao dos Modelos', fontweight='bold')
axes[0].legend(loc='lower right')

# Plot 2: Curvas Precision-Recall
for nome, (modelo, y_pred, y_proba) in modelos_treinados.items():
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    label_nome = nome.replace('_', ' ')
    axes[1].plot(rec, prec, lw=2.5, color=cores_modelos[nome],
                 label=f'{label_nome} (AP={ap:.4f})')

baseline = y_test.mean()
axes[1].axhline(y=baseline, color='k', linestyle='--', lw=1.5,
                label=f'Baseline ({baseline:.4f})')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Curva Precision-Recall', fontweight='bold')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('../docs/eval_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrizes de confusao para todos os modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nome, (modelo, y_pred, y_proba)) in zip(axes, modelos_treinados.items()):
    cm = confusion_matrix(y_test, y_pred)
    
    # Calcula percentuais
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Anotacoes combinando valor absoluto e percentual
    annot = np.array([[f'{v}\n({p:.1f}%)' for v, p in zip(row_v, row_p)]
                      for row_v, row_p in zip(cm, cm_pct)])
    
    sns.heatmap(
        cm_pct, annot=annot, fmt='', cmap='RdYlGn_r',
        xticklabels=['Adimplente', 'Inadimplente'],
        yticklabels=['Adimplente', 'Inadimplente'],
        ax=ax, vmin=0, vmax=100, linewidths=0.5
    )
    
    nome_label = nome.replace('_', ' ')
    auc = roc_auc_score(y_test, y_proba)
    ax.set_title(f'{nome_label}\nAUC={auc:.4f}', fontweight='bold')
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusao (% por linha)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eval_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('Leitura da matriz:')
print('  Superior esquerdo: acertos em Adimplentes (TN)')
print('  Inferior direito : acertos em Inadimplentes (TP)')
print('  Superior direito : adimplentes classificados como inadimplentes (FP)')
print('  Inferior esquerdo: inadimplentes classificados como adimplentes (FN) - o pior!')

In [ ]:
# KS Statistic Plot: separacao das distribuicoes de score
# Quanto maior a separacao entre as duas curvas, melhor o modelo

# Usa o melhor modelo (XGBoost)
nome_melhor = max(resultados, key=lambda x: x['AUC-ROC'])['Modelo'].replace(' ', '_')
_, y_pred_best, y_proba_best = modelos_treinados[nome_melhor]

scores_adim = np.sort(y_proba_best[y_test == 0])
scores_inad = np.sort(y_proba_best[y_test == 1])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribuicao dos scores
axes[0].hist(scores_adim, bins=50, alpha=0.6, color='#2ecc71',
             label='Adimplente', density=True)
axes[0].hist(scores_inad, bins=50, alpha=0.6, color='#e74c3c',
             label='Inadimplente', density=True)
axes[0].set_xlabel('Score de Probabilidade de Inadimplencia')
axes[0].set_ylabel('Densidade')
axes[0].set_title(f'Distribuicao dos Scores ({nome_melhor.replace("_", " ")})',
                  fontweight='bold')
axes[0].legend()

# KS Plot (distribuicoes acumuladas)
sorted_scores = np.sort(np.concatenate([scores_adim, scores_inad]))
cdf_adim = np.searchsorted(scores_adim, sorted_scores, side='right') / len(scores_adim)
cdf_inad = np.searchsorted(scores_inad, sorted_scores, side='right') / len(scores_inad)

axes[1].plot(sorted_scores, cdf_adim, color='#2ecc71', lw=2, label='Adimplente')
axes[1].plot(sorted_scores, cdf_inad, color='#e74c3c', lw=2, label='Inadimplente')

# Ponto de maxima separacao (KS)
ks_stat, _ = ks_2samp(scores_inad, scores_adim)
idx_max = np.argmax(np.abs(cdf_adim - cdf_inad))
x_ks = sorted_scores[idx_max]
axes[1].vlines(x_ks, cdf_adim[idx_max], cdf_inad[idx_max],
               color='navy', lw=2.5, linestyle='--',
               label=f'KS = {ks_stat:.4f}')

axes[1].set_xlabel('Score de Probabilidade')
axes[1].set_ylabel('Probabilidade Acumulada')
axes[1].set_title('KS Statistic - Separacao das Distribuicoes', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/eval_ks_statistic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'KS Statistic do {nome_melhor}: {ks_stat:.4f}')
print('Interpretacao:')
print('  KS > 0.40: Excelente separacao')
print('  KS > 0.30: Boa separacao')
print('  KS > 0.20: Separacao aceitavel')
print('  KS < 0.20: Separacao fraca')

## 6. Importancia das Features e Interpretabilidade

Um modelo nao pode ser uma caixa preta em credito. O Banco Central exige que as decisoes de concessao possam ser explicadas. Por isso, analisamos quais variaveis mais influenciam o modelo.

In [ ]:
# Importancia das features do XGBoost (melhor modelo)
xgb_model = modelos_treinados['XGBoost'][0]

importancias = pd.Series(
    xgb_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

top_n = 15
top_features = importancias.head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))
cores = plt.cm.RdYlGn(np.linspace(0.15, 0.85, top_n))

bars = ax.barh(
    range(top_n),
    top_features.values[::-1],
    color=cores[::-1]
)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features.index[::-1])
ax.set_xlabel('Importancia (Ganho de Informacao)')
ax.set_title(f'Top {top_n} Features Mais Importantes - XGBoost',
             fontweight='bold', fontsize=14)

for bar, val in zip(bars, top_features.values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../docs/eval_xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Interpretacao das features mais importantes:')
print('  loan_percent_income : quanto maior o comprometimento de renda, maior o risco')
print('  loan_int_rate        : taxas altas sinalizam clientes de maior risco')
print('  loan_grade_encoded   : a classificacao de risco ja captura muito')
print('  inadimplencia_previa : historico e um forte preditor de comportamento futuro')

## 7. Escolha e Salvamento do Melhor Modelo

In [ ]:
# Identifica o melhor modelo por AUC-ROC
df_res = pd.DataFrame(resultados)
melhor_nome_display = df_res.loc[df_res['AUC-ROC'].idxmax(), 'Modelo']
melhor_nome_key = melhor_nome_display.replace(' ', '_')
melhor_auc = df_res['AUC-ROC'].max()

print(f'Melhor modelo: {melhor_nome_display}')
print(f'AUC-ROC      : {melhor_auc:.4f}')

# Salva o melhor modelo
melhor_modelo = modelos_treinados[melhor_nome_key][0]
with open(MODELS_PATH / 'best_model.pkl', 'wb') as f:
    pickle.dump(melhor_modelo, f)
    
# Salva todos os modelos individualmente
for nome, (modelo, _, _) in modelos_treinados.items():
    with open(MODELS_PATH / f'{nome}.pkl', 'wb') as f:
        pickle.dump(modelo, f)

print('\nModelos salvos:')
for f in sorted(MODELS_PATH.glob('*.pkl')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}: {size_kb:.1f} KB')

## 8. Insights de Negocio

A parte mais importante: o que os dados nos dizem sobre o negocio?

Nao basta ter um modelo com AUC alto. Precisamos traduzir os resultados em linguagem de negocio.

In [ ]:
# Analise de threshold: qual o ponto de corte ideal?
# Em credito, podemos ajustar o threshold para equilibrar
# entre negar credito a bons clientes vs aprovar maus pagadores

_, _, y_proba_best = modelos_treinados[melhor_nome_key]

thresholds = np.arange(0.1, 0.9, 0.05)
resultados_threshold = []

for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    if y_pred_t.sum() > 0 and (1 - y_pred_t).sum() > 0:
        f1 = f1_score(y_test, y_pred_t)
        prec = precision_score(y_test, y_pred_t, zero_division=0)
        rec = recall_score(y_test, y_pred_t, zero_division=0)
        aprovacao = (y_pred_t == 0).mean() * 100  # taxa de aprovacao de credito
        resultados_threshold.append({
            'Threshold': round(t, 2),
            'F1': round(f1, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'Taxa Aprovacao (%)': round(aprovacao, 1)
        })

df_thresh = pd.DataFrame(resultados_threshold)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(df_thresh['Threshold'], df_thresh['F1'], 'b-o', label='F1-Score', lw=2)
axes[0].plot(df_thresh['Threshold'], df_thresh['Precision'], 'g-s', label='Precision', lw=2)
axes[0].plot(df_thresh['Threshold'], df_thresh['Recall'], 'r-^', label='Recall', lw=2)
axes[0].set_xlabel('Threshold de Decisao')
axes[0].set_ylabel('Score')
axes[0].set_title('Metricas por Threshold', fontweight='bold')
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Threshold padrao (0.5)')
axes[0].legend()

axes[1].plot(df_thresh['Threshold'], df_thresh['Taxa Aprovacao (%)'], 'purple', lw=2.5)
axes[1].fill_between(df_thresh['Threshold'], df_thresh['Taxa Aprovacao (%)'],
                      alpha=0.2, color='purple')
axes[1].set_xlabel('Threshold de Decisao')
axes[1].set_ylabel('Taxa de Aprovacao de Credito (%)')
axes[1].set_title('Impacto do Threshold na Taxa de Aprovacao', fontweight='bold')
axes[1].axvline(x=0.5, color='gray', linestyle='--', alpha=0.7)

plt.suptitle('Analise de Threshold: Trade-off entre Risco e Volume', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eval_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight de Negocio: O Threshold e uma decisao de negocio, nao tecnica!')
print('Threshold alto (ex: 0.7): menos falsos positivos, maior rigor, menor volume de credito')
print('Threshold baixo (ex: 0.3): mais credito concedido, maior tolerancia ao risco')

In [ ]:
print('=' * 65)
print('RESULTADO FINAL')
print('=' * 65)

print(f'\nMelhor modelo: {melhor_nome_display}')
print('\nMetricas no conjunto de teste:')
melhor_resultado = df_res[df_res['Modelo'] == melhor_nome_display].iloc[0]
for metrica in ['AUC-ROC', 'F1-Score', 'Precision', 'Recall', 'KS Statistic']:
    print(f'  {metrica:15s}: {melhor_resultado[metrica]:.4f}')

print('\nInsights de Negocio Validados pelo Modelo:')
print('  1. loan_grade e o preditor mais forte: cada nivel de A a G')
print('     aumenta significativamente a taxa de inadimplencia')
print('  2. Comprometimento de renda acima de 30% e um sinal de alerta critico')
print('  3. Inadimplencia previa multiplica por 2.3x o risco atual')
print('  4. Taxa de juros alta e tanto causa quanto efeito do risco')
print('  5. Emprestimos para consolidacao de divida tem o maior risco')
print('     de inadimplencia entre todas as finalidades')

print('\nProximo passo:')
print('  -> Notebook 04: Azure + Databricks (ambiente de producao na nuvem)')
print('=' * 65)

---

Modelos treinados, avaliados e salvos! No **Notebook 04** vamos ver como escalar tudo isso para a nuvem usando Azure e Databricks.

**Nayane Araujo** | [GitHub](https://github.com/Nayanearaujo)